# Install Dependencies

In [ ]:
!pip install sentence-transformers transformers peft accelerate  faiss-cpu pandas numpy


In [ ]:
!pip install -U datasets


In [ ]:
!ls


In [ ]:
import datasets

# Mount Google Drive and Load Dataset

In [ ]:
from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define dataset path
DATA_FOLDER = "/content/drive/MyDrive/mutual_funds_data"
file_path = os.path.join(DATA_FOLDER, "data_with_sub_categories.csv")

df = pd.read_csv(file_path)
print("✅ Dataset Loaded Successfully!")

print(df.head())

In [ ]:
import requests
import concurrent.futures

# Function to fetch NAV for a given scheme
def fetch_nav(scheme_code):
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    try:
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            data = response.json()
            if "data" in data and len(data["data"]) > 0:
                return float(data["data"][0]["nav"])
    except requests.RequestException as e:
        print(f"❌ Error fetching NAV for {scheme_code}: {e}")
    return None

# Fetch NAV values in parallel
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    nav_values = list(executor.map(fetch_nav, df["Scheme Codes"]))

df["NAV"] = nav_values

# Save updated dataset
updated_nav_path = os.path.join(DATA_FOLDER, "updated_data_with_nav.csv")
df.to_csv(updated_nav_path, index=False)
print(f"✅ NAV values updated and saved at: {updated_nav_path}")


In [ ]:
from google.colab import drive
import os
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# File paths
DATA_FOLDER = "/content/drive/MyDrive/mutual_funds_data"
CSV_FILE = os.path.join(DATA_FOLDER, "updated_data_with_nav.csv")

# Load dataset
df = pd.read_csv(CSV_FILE)
print("✅ Dataset Loaded")


# pre trained model

In [ ]:
!pip install wandb

In [ ]:
import torch
torch.cuda.is_available()



In [ ]:
# !pip uninstall -y bitsandbytes
!pip install git+https://github.com/TimDettmers/bitsandbytes.git


In [ ]:
# 📘 AUTOMATED RAG PIPELINE WITH FINE-TUNED PHI-2 (GOOGLE COLAB READY)


# 📂 STEP 2: MOUNT GOOGLE DRIVE & LOAD DATASET
from google.colab import drive
import os
import pandas as pd

# Mount Drive
drive.mount('/content/drive')

# File Paths
DATA_FOLDER = "/content/drive/MyDrive/mutual_funds_data"
CSV_FILE = os.path.join(DATA_FOLDER, "data_with_sub_categories.csv")
UPDATED_CSV = os.path.join(DATA_FOLDER, "updated_data_with_nav.csv")

# Load Dataset
df = pd.read_csv(CSV_FILE)
print("✅ Original Dataset Loaded")
# STEP 4: CONVERT CSV TO JSONL FOR FINE-TUNING
import json

jsonl_path = "/content/mutual_funds.jsonl"
df = pd.read_csv(UPDATED_CSV)
df.fillna({"returns_1yr": 0, "returns_3yr": 0, "returns_5yr": 0, "expense_ratio": 0}, inplace=True)

with open(jsonl_path, "w") as f:
    for _, row in df.iterrows():
        prompt = (
            f"User goal: I'm looking for a mutual fund in the '{row['Sub_Category']}' category "
            f"with '{row['risk_level']}' risk level. Explain if the following fund is a suitable option:\n\n"
            f"Scheme Name: {row['scheme_name']}\nNAV: ₹{row['NAV']}\n"
            f"1Y Return: {row['returns_1yr']}%\n3Y Return: {row['returns_3yr']}%\n5Y Return: {row['returns_5yr']}%\n"
            f"Expense Ratio: {row['expense_ratio']}%\nRisk Level: {row['risk_level']}"
        )
        response = (
            f"'{row['scheme_name']}' offers a {row['risk_level']} risk profile, suitable for investors interested in the '{row['Sub_Category']}' category. "
            f"With steady returns and an expense ratio of {row['expense_ratio']}%, it supports long-term growth."
        )
        f.write(json.dumps({"prompt": prompt, "response": response}) + "\n")
print(f"✅ JSONL created: {jsonl_path}")

# STEP 5: CPU-COMPATIBLE FINE-TUNING (SAFE FOR NON-CUDA ENVIRONMENTS)
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

# Load dataset
dataset = load_dataset("json", data_files=jsonl_path, split="train")

# Load tokenizer and 4-bit quantized model
model_id = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16
# )
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     device_map="auto",
#     quantization_config=bnb_config,
#     trust_remote_code=True
# )

# Apply LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

# Tokenize
def tokenize(sample):
    full = f"{sample['prompt']}\n\nAnswer: {sample['response']}"
    return tokenizer(full, padding="max_length", truncation=True, max_length=512)

tokenized_ds = dataset.map(tokenize)

# Training Args (GPU-Optimized)
training_args = TrainingArguments(
    output_dir="/content/phi2-finetuned-mutual-funds",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    fp16=True,
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    save_total_limit=1,
    logging_dir="/content/logs"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)
model.enable_input_require_grads()
model.print_trainable_parameters()

# Train and Save
trainer.train()

SAVE_PATH = os.path.join(DATA_FOLDER, "fine_tuned_phi2")
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Fine-tuned model saved at: {SAVE_PATH}")



# Embeddings Genrate and  FAISS Index

In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

df["fund_description"] = df["scheme_name"] + " - " + df["Sub_Category"]
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["fund_description"].tolist(), show_progress_bar=True)

np.save(os.path.join(DATA_FOLDER, "fund_embeddings.npy"), embeddings)
df["embedding_index"] = range(len(df))
df.to_csv(os.path.join(DATA_FOLDER, "updated_data_with_embeddings.csv"), index=False)

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)
faiss.write_index(index, os.path.join(DATA_FOLDER, "fund_faiss.index"))

#  load dataset & index for retrieval

In [ ]:
updated_csv = os.path.join(DATA_FOLDER, "updated_data_with_embeddings.csv")
df_embeddings = pd.read_csv(updated_csv)
embeddings = np.load(os.path.join(DATA_FOLDER, "fund_embeddings.npy"))
index = faiss.read_index(os.path.join(DATA_FOLDER, "fund_faiss.index"))

# live upddate of NAV

In [ ]:
import concurrent.futures


In [ ]:
def update_navs_live():
    print(" Fetching live NAVs...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        navs = list(executor.map(fetch_nav, df_embeddings["Scheme Codes"]))
    df_embeddings["NAV"] = navs
    print("✅ Live NAVs updated.")


In [ ]:
update_navs_live()

# user input and ranking

In [ ]:
def get_user_input():
    categories = df_embeddings["Sub_Category"].unique()
    category_dict = {str(i + 1): cat for i, cat in enumerate(categories)}

    print("\n📌 Select Mutual Fund Category (by number or name):")
    for k, v in category_dict.items():
        print(f"{k}. {v}")

    while True:
        val = input("Enter category number or name: ").strip()
        if val.isdigit() and val in category_dict:
            selected_category = category_dict[val]
            break
        elif val.lower() in map(str.lower, categories):
            selected_category = next(cat for cat in categories if cat.lower() == val.lower())
            break
        else:
            print("❌ Invalid category. Please choose from the list above.")

    risk_levels = ["Low", "Medium", "High"]
    while True:
        risk = input("Select Risk Level (Low / Medium / High): ").strip().capitalize()
        if risk in risk_levels:
            break
        print("❌ Invalid. Choose Low, Medium, or High.")

    # Provide example queries + handle vague input
    print("\n💬 Describe your investment goal. Examples:")
    print(" - I want long-term growth with moderate risk.")
    print(" - I prefer safe investments with steady returns.")
    print(" - I'm okay with high risk for higher returns.")

    vague_keywords = ["best", "any", "good", "don't know", "whatever", "idk", "anything"]
    while True:
        goal = input("Describe your investment goal: ").strip()
        if len(goal) < 10 or any(vague in goal.lower() for vague in vague_keywords):
            print("⚠️ Please provide a more specific goal like:")
            print("   - 'I want consistent growth for 5 years'")
            print("   - 'I prefer stable income with low risk'")
        else:
            break

    return selected_category, risk, goal


In [ ]:
def normalize_column(df, col):
    return (df[col] - df[col].min()) / (df[col].max() - df[col].min())

In [ ]:
def keyword_filter(query_text):
    query_lower = query_text.lower()
    matches = df_embeddings[
        df_embeddings["scheme_name"].astype(str).str.lower().str.contains(query_lower) |
        df_embeddings["Sub_Category"].astype(str).str.lower().str.contains(query_lower) |
        df_embeddings["risk_level"].astype(str).str.lower().str.contains(query_lower)
    ]
    return matches["embedding_index"].tolist()


In [ ]:
def rank_funds(indices, distances):
    selected = df_embeddings.iloc[indices].copy()

    # Normalize features
    nav_score = normalize_column(selected, "NAV")
    exp_score = 1 - normalize_column(selected, "expense_ratio")
    r1 = normalize_column(selected, "returns_1yr")
    r3 = normalize_column(selected, "returns_3yr")
    r5 = normalize_column(selected, "returns_5yr")

    # Composite return score
    return_score = (r1 * 0.3 + r3 * 0.3 + r5 * 0.4)

    # Final score
    final_score = (
        np.array(distances[:len(selected)]) * 0.4 +
        nav_score * 0.2 +
        exp_score * 0.15 +
        return_score * 0.25
    )

    selected["final_score"] = final_score
    selected["nav_score"] = nav_score
    selected["exp_score"] = exp_score
    selected["return_score"] = return_score

    return selected.sort_values("final_score", ascending=False).head(5)


In [ ]:
def rewrite_query(user_input):
    vague_keywords = ["best", "good", "any", "anything", "whatever", "top", "idk", "don't know", "no idea", "you decide", "choose for me"]

    # 🔍 Detect vague input
    if len(user_input.strip()) < 10 or any(word in user_input.lower() for word in vague_keywords):
        print("⚠️ Vague input detected. Rewriting to make it more specific...")

        vague_prompt = (
            f"The user gave a vague goal: '{user_input}'. Rewrite it as a specific mutual fund investment goal. "
            f"Add realistic constraints like long-term growth, stable income, or low risk."
        )
        response = llm(vague_prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"]
        rewritten = response.replace(vague_prompt, "").strip()
        print(f"✅ Rewritten vague goal: {rewritten}")
        return rewritten

    # 🔁 For normal, clear queries — rewrite as usual
    prompt = (
        f"You are a financial assistant. Rephrase the user's goal into a detailed mutual fund search query.\n"
        f"User goal: {user_input}\n"
        f"Rewritten Query:"
    )
    response = llm(prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"]
    rewritten = response.replace(prompt, "").strip()

    print(f"🧠 Rewritten Query for FAISS: {rewritten}")
    return rewritten


In [ ]:
from sentence_transformers import SentenceTransformer

retrieval_model = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
def search_funds():
    cat, risk, query = get_user_input()

    # Step 1: Rephrase query using LLM
    rewritten_query = rewrite_query(query)

    # Step 2: Semantic search via FAISS using the correct model
    q_embed = retrieval_model.encode([rewritten_query])  # ✅ Fixed here
    distances, indices = index.search(q_embed, 10)
    faiss_indices = list(indices[0])

    # Step 3: Keyword-based filtering
    keyword_indices = keyword_filter(rewritten_query)

    # Step 4: Combine FAISS and keyword results (preserve order, no duplicates)
    combined_indices = list(dict.fromkeys(keyword_indices + faiss_indices))

    # Step 5: Show internal match lists
    print(f" FAISS matches: {faiss_indices}")
    print(f" Keyword matches: {keyword_indices}")
    print(f"Combined top indices: {combined_indices[:5]}")

    # Step 6: Filter by selected category and risk
    filtered = [
        idx for idx in combined_indices
        if df_embeddings.iloc[idx]["Sub_Category"] == cat and df_embeddings.iloc[idx]["risk_level"] == risk
    ]

    if not filtered:
        # print(" No exact match. Showing closest from combined.")
        filtered = combined_indices

    return rank_funds(filtered, distances[0]), query


# load fine tuned model

In [ ]:
import os
os.listdir("/content/drive/MyDrive/mutual_funds_data/fine_tuned_phi2")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
import torch
import os

# Fine-tuned model path
llm_path = os.path.join(DATA_FOLDER, "fine_tuned_phi2")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(llm_path)

# Load base model (phi-2)
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",  # base model
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Load LoRA adapter on top of base model
model = PeftModel.from_pretrained(base_model, llm_path)

# Set padding token if needed
tokenizer.pad_token = tokenizer.eos_token

# Create pipeline
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)




# explanations


In [ ]:
def explain_funds(funds, user_query):
    print("\n🔹 AI-Based Financial Explanations:\n")
    for idx, (_, row) in enumerate(funds.iterrows(), 1):
        prompt = (
            f"User goal: {user_query}\n"
            f"Scheme: {row['scheme_name']}\n"
            f"NAV: ₹{row['NAV']}\n"
            f"Returns: {row['returns_1yr']}% (1Y), {row['returns_3yr']}% (3Y), {row['returns_5yr']}% (5Y)\n"
            f"Expense Ratio: {row['expense_ratio']}%\n"
            f"Risk Level: {row['risk_level']}\n"
            f"Explain why this fund is a good match."
        )
        output = llm(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
        explanation = output.replace(prompt, "").strip()
        print(f"🔸 {idx}. {row['scheme_name']}")
        print(f"   {explanation}\n")

def show_score_breakdown(funds):
    print("\n Fund Scoring Breakdown:")
    for idx, (_, row) in enumerate(funds.iterrows(), 1):
        print(f"🔸 {idx}. {row['scheme_name']}")
        print(f"    NAV Score: {row['nav_score']:.2f}")
        print(f"    Expense Ratio Score: {row['exp_score']:.2f}")
        print(f"    Return Score (1Y/3Y/5Y combined): {row['return_score']:.2f}")
        print(f"    Final Score: {row['final_score']:.2f}\n")


# run the system

In [ ]:
recommended_funds, user_query = search_funds()
print("\n🔹 Recommended Mutual Funds (Top 5):")
print(recommended_funds[["scheme_name", "NAV", "returns_1yr", "returns_3yr", "returns_5yr", "expense_ratio"]])
explain_funds(recommended_funds, user_query)
show_score_breakdown(recommended_funds)

plot_nav_and_returns(recommended_funds)


# New Section

# charts addition

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
def plot_nav_and_returns(funds):
    names = funds["scheme_name"]
    navs = funds["NAV"]
    r1 = funds["returns_1yr"]
    r3 = funds["returns_3yr"]
    r5 = funds["returns_5yr"]

    # 📊 NAV bar chart
    plt.figure(figsize=(10, 5))
    plt.bar(names, navs, color='steelblue')
    plt.xticks(rotation=45, ha='right')
    plt.title("NAV Comparison")
    plt.ylabel("NAV (₹)")
    plt.tight_layout()
    plt.show()

    # 📊 Returns comparison chart
    x = range(len(names))
    width = 0.25

    plt.figure(figsize=(10, 5))
    plt.bar([i - width for i in x], r1, width=width, label="1Y Return")
    plt.bar(x, r3, width=width, label="3Y Return")
    plt.bar([i + width for i in x], r5, width=width, label="5Y Return")

    plt.xticks(x, names, rotation=45, ha='right')
    plt.ylabel("Return (%)")
    plt.title("Returns Comparison (1Y / 3Y / 5Y)")
    plt.legend()
    plt.tight_layout()
    plt.show()


# frontend


In [ ]:
# !pip install gradio



In [ ]:
# import gradio as gr

# # 🔌 Backend API function
# def recommend(goal, category, risk):
#     rewritten_query = rewrite_query(goal)
#     q_embed = model.encode([rewritten_query])
#     distances, indices = index.search(q_embed, 10)

#     filtered = [idx for idx in indices[0] if df_embeddings.iloc[idx]["Sub_Category"] == category and df_embeddings.iloc[idx]["risk_level"] == risk]
#     if not filtered:
#         filtered = indices[0]

#     top_funds = rank_funds(filtered, distances[0])

#     results = []
#     for _, row in top_funds.iterrows():
#         prompt = (
#             f"User goal: {goal}\n"
#             f"Scheme: {row['scheme_name']}\nNAV: ₹{row['NAV']}\n"
#             f"Returns: {row['returns_1yr']}% (1Y), {row['returns_3yr']}% (3Y), {row['returns_5yr']}% (5Y)\n"
#             f"Expense Ratio: {row['expense_ratio']}%\nRisk Level: {row['risk_level']}\n"
#             f"Explain why this fund is a good match."
#         )
#         response = llm(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
#         explanation = response.replace(prompt, "").strip()
#         results.append({
#             "scheme_name": row["scheme_name"],
#             "NAV": row["NAV"],
#             "returns_1yr": row["returns_1yr"],
#             "returns_3yr": row["returns_3yr"],
#             "returns_5yr": row["returns_5yr"],
#             "expense_ratio": row["expense_ratio"],
#             "risk_level": row["risk_level"],
#             "explanation": explanation
#         })

#     return results  # ⬅ return plain list — Gradio will wrap it under 'data' automatically

# # ✅ Make sure input type is array-based and order is correct
# demo = gr.Interface(
#     fn=recommend,
#     inputs=[gr.Text(label="Goal"), gr.Text(label="Category"), gr.Text(label="Risk")],
#     outputs="json"
# )

# demo.launch(share=True)


In [ ]:
# !pip install fastapi uvicorn nest-asyncio pyngrok

# from fastapi import FastAPI
# from pydantic import BaseModel
# from fastapi.middleware.cors import CORSMiddleware
# import nest_asyncio
# from pyngrok import ngrok

# app = FastAPI()

# # CORS for frontend access
# app.add_middleware(
#     CORSMiddleware,
#     allow_origins=["*"],
#     allow_methods=["*"],
#     allow_headers=["*"],
# )

# class QueryInput(BaseModel):
#     goal: str
#     category: str
#     risk: str

# @app.post("/recommend")
# def recommend_funds(query: QueryInput):
#     rewritten_query = rewrite_query(query.goal)
#     q_embed = model.encode([rewritten_query])
#     distances, indices = index.search(q_embed, 10)

#     filtered = [idx for idx in indices[0] if df_embeddings.iloc[idx]["Sub_Category"] == query.category and df_embeddings.iloc[idx]["risk_level"] == query.risk]
#     if not filtered:
#         filtered = indices[0]

#     top_funds = rank_funds(filtered, distances[0])
#     results = []
#     for _, row in top_funds.iterrows():
#         prompt = (
#             f"User goal: {query.goal}\n"
#             f"Scheme: {row['scheme_name']}\nNAV: ₹{row['NAV']}\n"
#             f"Returns: {row['returns_1yr']}% (1Y), {row['returns_3yr']}% (3Y), {row['returns_5yr']}% (5Y)\n"
#             f"Expense Ratio: {row['expense_ratio']}%\nRisk Level: {row['risk_level']}\n"
#             f"Explain why this fund is a good match."
#         )
#         output = llm(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
#         explanation = output.replace(prompt, "").strip()

#         results.append({
#             "scheme_name": row["scheme_name"],
#             "NAV": row["NAV"],
#             "returns_1yr": row["returns_1yr"],
#             "returns_3yr": row["returns_3yr"],
#             "returns_5yr": row["returns_5yr"],
#             "expense_ratio": row["expense_ratio"],
#             "risk_level": row["risk_level"],
#             "explanation": explanation
#         })

#     return {"funds": results}

# # Start FastAPI
# nest_asyncio.apply()
# public_url = ngrok.connect(8000)
# print(f"🌐 Public URL: {public_url}")

# import uvicorn
# uvicorn.run(app, port=8000)


In [ ]:
!pip install fastapi uvicorn nest-asyncio pyngrok

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# 🎯 FastAPI app
app = FastAPI()

# 🌐 Enable CORS for React
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # You can restrict to specific domain later
    allow_methods=["*"],
    allow_headers=["*"],
)

# 📦 Data model for frontend request
class QueryInput(BaseModel):
    goal: str
    category: str
    risk: str


In [ ]:
@app.post("/recommend")
def recommend_funds(query: QueryInput):
    cat = query.category
    risk = query.risk
    user_goal = query.goal

    rewritten_query = rewrite_query(user_goal)
    q_embed = model.encode([rewritten_query])
    distances, indices = index.search(q_embed, 10)
    faiss_indices = list(indices[0])
    keyword_indices = keyword_filter(rewritten_query)

    combined_indices = list(dict.fromkeys(keyword_indices + faiss_indices))

    filtered = [
        idx for idx in combined_indices
        if df_embeddings.iloc[idx]["Sub_Category"] == cat and df_embeddings.iloc[idx]["risk_level"] == risk
    ]
    if not filtered:
        filtered = combined_indices

    top_funds = rank_funds(filtered, distances[0])

    results = []
    for _, row in top_funds.iterrows():
        prompt = (
            f"User goal: {user_goal}\n"
            f"Scheme: {row['scheme_name']}\nNAV: ₹{row['NAV']}\n"
            f"Returns: {row['returns_1yr']}% (1Y), {row['returns_3yr']}% (3Y), {row['returns_5yr']}% (5Y)\n"
            f"Expense Ratio: {row['expense_ratio']}%\nRisk Level: {row['risk_level']}\n"
            f"Explain why this fund is a good match."
        )
        output = llm(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
        explanation = output.replace(prompt, "").strip()

        results.append({
            "scheme_name": row["scheme_name"],
            "NAV": row["NAV"],
            "returns_1yr": row["returns_1yr"],
            "returns_3yr": row["returns_3yr"],
            "returns_5yr": row["returns_5yr"],
            "expense_ratio": row["expense_ratio"],
            "risk_level": row["risk_level"],
            "explanation": explanation
        })

    return {"funds": results}


In [ ]:
from pyngrok import conf, ngrok

# Paste your authtoken here between the quotes
conf.get_default().auth_token = "2tMP1YSN9d1GlZIidmiiquhQoiA_3YyNENe7JFVCx91Wh3ypr"


In [ ]:
import nest_asyncio
import uvicorn
from pyngrok import ngrok

nest_asyncio.apply()

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"🌐 Public URL: {public_url}")
@app.get("/categories")
def get_categories():
    unique_categories = df_embeddings["Sub_Category"].dropna().unique().tolist()
    return {"categories": unique_categories}


# Run FastAPI app
uvicorn.run(app, port=8000)

# Testing the model

In [ ]:
!pip install evaluate bert_score rouge_score


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

model_path = "/content/drive/MyDrive/mutual_funds_data/fine_tuned_phi2"

# Load tokenizer and base model
tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, model_path)

# Create pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)


In [ ]:
import json

jsonl_path = "/content/mutual_funds.jsonl"
samples = []
with open(jsonl_path, "r") as f:
    for i, line in enumerate(f):
        if i >= 20: break
        entry = json.loads(line)
        samples.append((entry["prompt"], entry["response"]))


In [ ]:
# STEP 1: (5 samples only for speed)
import json
sample_prompts, sample_refs, sample_preds = [], [], []

jsonl_path = "/content/mutual_funds.jsonl"
with open(jsonl_path, "r") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        entry = json.loads(line)
        sample_prompts.append(entry["prompt"])
        sample_refs.append(entry["response"])

# STEP 2: Run Fast LLM Generation
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

for prompt in sample_prompts:
    output = pipe(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"]
    gen = output.replace(prompt, "").strip()
    sample_preds.append(gen)

# STEP 3: BLEU (Fast and Valid Metric)
import evaluate
bleu = evaluate.load("bleu")
result = bleu.compute(predictions=sample_preds, references=[[r] for r in sample_refs])
print(f"\n✅ Quick BLEU Score: {result['bleu']:.4f}")

# STEP 4: Show Results
import pandas as pd
df_eval = pd.DataFrame({
    "Prompt": sample_prompts,
    "Reference": sample_refs,
    "Generated": sample_preds
})
df_eval.head()


# New Section

In [ ]:
# ====================== 🔧 FIX sympy issue ======================
!pip uninstall -y sympy -q
!pip install -q sympy==1.12

# ====================== 📦 Install All Requirements ======================
!pip install -q sentence-transformers transformers peft accelerate faiss-cpu pandas numpy fastapi uvicorn nest-asyncio pyngrok

# ====================== 📂 Load Data ======================
from google.colab import drive
import os, pandas as pd, numpy as np, faiss, torch, nest_asyncio
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf

# Mount Google Drive
drive.mount('/content/drive')

# Paths
DATA_FOLDER = "/content/drive/MyDrive/mutual_funds_data"
df_embeddings = pd.read_csv(os.path.join(DATA_FOLDER, "updated_data_with_embeddings.csv"))
embeddings = np.load(os.path.join(DATA_FOLDER, "fund_embeddings.npy"))
index = faiss.read_index(os.path.join(DATA_FOLDER, "fund_faiss.index"))

# ====================== 🧠 Load Models ======================
retrieval_model = SentenceTransformer("all-MiniLM-L6-v2")

MODEL_PATH = os.path.join(DATA_FOLDER, "fine_tuned_phi2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

# ====================== 🚀 FastAPI Setup ======================
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class QueryInput(BaseModel):
    goal: str
    category: str
    risk: str

# ====================== 🧮 Logic Functions ======================
def normalize_column(df, col):
    return (df[col] - df[col].min()) / (df[col].max() - df[col].min())

def keyword_filter(query_text):
    q = query_text.lower()
    matches = df_embeddings[
        df_embeddings["scheme_name"].astype(str).str.lower().str.contains(q) |
        df_embeddings["Sub_Category"].astype(str).str.lower().str.contains(q) |
        df_embeddings["risk_level"].astype(str).str.lower().str.contains(q)
    ]
    return matches["embedding_index"].tolist()

def rank_funds(indices, distances):
    selected = df_embeddings.iloc[indices].copy()
    nav_score = normalize_column(selected, "NAV")
    exp_score = 1 - normalize_column(selected, "expense_ratio")
    r1 = normalize_column(selected, "returns_1yr")
    r3 = normalize_column(selected, "returns_3yr")
    r5 = normalize_column(selected, "returns_5yr")
    return_score = (r1 * 0.3 + r3 * 0.3 + r5 * 0.4)
    final_score = (np.array(distances[:len(selected)]) * 0.4 +
                   nav_score * 0.2 +
                   exp_score * 0.15 +
                   return_score * 0.25)
    selected["final_score"] = final_score
    return selected.sort_values("final_score", ascending=False).head(5)

def rewrite_query(user_input):
    vague = ["best", "any", "good", "whatever", "top", "idk", "don't know"]
    if len(user_input.strip()) < 10 or any(v in user_input.lower() for v in vague):
        vague_prompt = f"The user gave a vague goal: '{user_input}'. Rewrite it with realistic constraints."
        rewritten = llm(vague_prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"]
        return rewritten.replace(vague_prompt, "").strip()
    prompt = f"You are a financial assistant. Rephrase the user's goal into a detailed mutual fund search query.\nUser goal: {user_input}\nRewritten Query:"
    rewritten = llm(prompt, max_new_tokens=60, do_sample=False)[0]["generated_text"]
    return rewritten.replace(prompt, "").strip()

# ====================== 🔍 Recommendation Endpoint ======================
@app.post("/recommend")
def recommend_funds(query: QueryInput):
    user_goal = query.goal
    category = query.category
    risk = query.risk
    rewritten_query = rewrite_query(user_goal)
    q_embed = retrieval_model.encode([rewritten_query])
    distances, indices = index.search(q_embed, 10)

    faiss_indices = list(indices[0])
    keyword_indices = keyword_filter(rewritten_query)
    combined = list(dict.fromkeys(keyword_indices + faiss_indices))

    filtered = [
        idx for idx in combined
        if df_embeddings.iloc[idx]["Sub_Category"] == category and df_embeddings.iloc[idx]["risk_level"] == risk
    ]
    if not filtered:
        filtered = combined

    top_funds = rank_funds(filtered, distances[0])
    results = []

    for _, row in top_funds.iterrows():
        prompt = (
            f"User goal: {user_goal}\n"
            f"Scheme: {row['scheme_name']}\nNAV: ₹{row['NAV']}\n"
            f"Returns: {row['returns_1yr']}% (1Y), {row['returns_3yr']}% (3Y), {row['returns_5yr']}% (5Y)\n"
            f"Expense Ratio: {row['expense_ratio']}%\nRisk Level: {row['risk_level']}\n"
            f"Explain why this fund is a good match."
        )
        explanation = llm(prompt, max_new_tokens=100, do_sample=False)[0]["generated_text"]
        results.append({
            "scheme_name": row["scheme_name"],
            "NAV": row["NAV"],
            "returns_1yr": row["returns_1yr"],
            "returns_3yr": row["returns_3yr"],
            "returns_5yr": row["returns_5yr"],
            "expense_ratio": row["expense_ratio"],
            "risk_level": row["risk_level"],
            "explanation": explanation.replace(prompt, "").strip()
        })

    return {"funds": results}

# ====================== 📊 Categories Endpoint ======================
@app.get("/categories")
def get_categories():
    return {"categories": df_embeddings["Sub_Category"].dropna().unique().tolist()}

# ====================== 🚀 Start FastAPI with Ngrok ======================
conf.get_default().auth_token = "2tMP1YSN9d1GlZIidmiiquhQoiA_3YyNENe7JFVCx91Wh3ypr"
public_url = ngrok.connect(8000)
print(f"🌐 Public URL: {public_url}")

nest_asyncio.apply()
import uvicorn
uvicorn.run(app, port=8000)


In [ ]:
!pip uninstall -y sympy
!pip install sympy==1.12 --force-reinstall --no-cache-dir


In [ ]:
import sympy
print(sympy.__version__)  # MUST be 1.12
print("✅ SymPy is clean and ready.")


In [ ]:
# ✅ Install clean dependencies
!pip uninstall -y sympy -q
!pip install -q sympy==1.12
!pip install -q sentence-transformers transformers peft accelerate faiss-cpu pandas numpy fastapi uvicorn nest-asyncio pyngrok

# ✅ Imports
import os, pandas as pd, numpy as np, faiss, torch, nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from peft import PeftModel

# ✅ Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ✅ Load data
DATA_FOLDER = "/content/drive/MyDrive/mutual_funds_data"
df_embeddings = pd.read_csv(os.path.join(DATA_FOLDER, "updated_data_with_embeddings.csv"))
embeddings = np.load(os.path.join(DATA_FOLDER, "fund_embeddings.npy"))
index = faiss.read_index(os.path.join(DATA_FOLDER, "fund_faiss.index"))

# ✅ Load and prepare model
retrieval_model = SentenceTransformer("all-MiniLM-L6-v2", device='cpu')

MODEL_PATH = os.path.join(DATA_FOLDER, "fine_tuned_phi2")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForCausalLM.from_pretrained("microsoft/phi-2", torch_dtype=torch.float16, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

# ✅ Move model to correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# base_model.to(device)
model.to(device)

# ✅ LLM generation function
def llm_generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ✅ FastAPI setup
app = FastAPI()

# ✅ CORS for frontend connection (e.g. React + ngrok)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Change this to your frontend URL in production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ✅ Input model
class QueryInput(BaseModel):
    goal: str
    category: str
    risk: str

# ✅ Helper functions
def normalize_column(df, col):
    return (df[col] - df[col].min()) / (df[col].max())

def keyword_filter(query_text):
    q = query_text.lower()
    matches = df_embeddings[
        df_embeddings["scheme_name"].astype(str).str.lower().str.contains(q) |
        df_embeddings["Sub_Category"].astype(str).str.lower().str.contains(q) |
        df_embeddings["risk_level"].astype(str).str.lower().str.contains(q)
    ]
    return matches["embedding_index"].tolist()

def rank_funds(indices, distances):
    selected = df_embeddings.iloc[indices].copy()
    nav_score = normalize_column(selected, "NAV")
    exp_score = 1 - normalize_column(selected, "expense_ratio")
    r1 = normalize_column(selected, "returns_1yr")
    r3 = normalize_column(selected, "returns_3yr")
    r5 = normalize_column(selected, "returns_5yr")
    return_score = (r1 * 0.3 + r3 * 0.3 + r5 * 0.4)
    final_score = (np.array(distances[:len(selected)]) * 0.4 +
                   nav_score * 0.2 +
                   exp_score * 0.15 +
                   return_score * 0.25)
    selected["final_score"] = final_score
    return selected.sort_values("final_score", ascending=False).head(5)

def rewrite_query(user_input):
    vague = ["best", "any", "good", "whatever", "top", "idk", "don't know"]
    if len(user_input.strip()) < 10 or any(v in user_input.lower() for v in vague):
        vague_prompt = f"The user gave a vague goal: '{user_input}'. Rewrite it with realistic constraints."
        return llm_generate(vague_prompt, max_new_tokens=60).replace(vague_prompt, "").strip()
    prompt = f"You are a financial assistant. Rephrase the user's goal into a detailed mutual fund search query.\nUser goal: {user_input}\nRewritten Query:"
    return llm_generate(prompt, max_new_tokens=60).replace(prompt, "").strip()

# ✅ /recommend endpoint
@app.post("/recommend")
def recommend_funds(query: QueryInput):
    user_goal = query.goal
    category = query.category
    risk = query.risk
    rewritten_query = rewrite_query(user_goal)
    q_embed = retrieval_model.encode([rewritten_query])
    distances, indices = index.search(q_embed, 10)

    faiss_indices = list(indices[0])
    keyword_indices = keyword_filter(rewritten_query)
    combined = list(dict.fromkeys(keyword_indices + faiss_indices))

    filtered = [
        idx for idx in combined
        if df_embeddings.iloc[idx]["Sub_Category"] == category and df_embeddings.iloc[idx]["risk_level"] == risk
    ]
    if not filtered:
        filtered = combined

    top_funds = rank_funds(filtered, distances[0])
    results = []

    for _, row in top_funds.iterrows():
        prompt = (
            f"User goal: {user_goal}\n"
            f"Scheme: {row['scheme_name']}\nNAV: ₹{row['NAV']}\n"
            f"Returns: {row['returns_1yr']}% (1Y), {row['returns_3yr']}% (3Y), {row['returns_5yr']}% (5Y)\n"
            f"Expense Ratio: {row['expense_ratio']}%\nRisk Level: {row['risk_level']}\n"
            f"Explain why this fund is a good match."
        )
        explanation = llm_generate(prompt, max_new_tokens=100).replace(prompt, "").strip()
        results.append({
            "scheme_name": row["scheme_name"],
            "NAV": row["NAV"],
            "returns_1yr": row["returns_1yr"],
            "returns_3yr": row["returns_3yr"],
            "returns_5yr": row["returns_5yr"],
            "expense_ratio": row["expense_ratio"],
            "risk_level": row["risk_level"],
            "explanation": explanation
        })

    return {"funds": results}

# ✅ /categories endpoint
@app.get("/categories")
def get_categories():
    return {"categories": df_embeddings["Sub_Category"].dropna().unique().tolist()}

# ✅ Launch server using ngrok
import uvicorn
conf.get_default().auth_token = "2tMP1YSN9d1GlZIidmiiquhQoiA_3YyNENe7JFVCx91Wh3ypr"
public_url = ngrok.connect(8000)
print(f"🌐 Public URL: {public_url}")
nest_asyncio.apply()
uvicorn.run(app, port=8000)
